# Exp 25: Coarse dropout, downsample 0.4

Card: [exp25_downsample_04.md](../../experiments/exp25_downsample_04.md)

In [ ]:
import os

import matplotlib.pyplot as plt
import mlflow
from sklearn.metrics import average_precision_score, precision_recall_curve

from core.config import CLASSES, MLFLOW_URI, ROOT
from core.cv import VIP, Config, CrossValidator, CVResult
from core.gradcam import GradCamFigures, compare

SMOKE = os.environ.get("SMOKE") == "1"
if SMOKE:
    OVERRIDES = {
        "experiment": "cnn-model-downsample-04-smoke",
        "epochs": 1,
        "img_size": 64,
        "n_other": 20,
        "num_workers": 0,
        "retrain": True,
        "log_every_n_steps": 1,
    }
    N_SAMPLE = 2  # Cat, Dog images per comparison figure
    N_POSITIVES = 2  # images per VIP Grad-CAM figure
else:
    OVERRIDES = {}
    N_SAMPLE = 16  # exp19 sample
    N_POSITIVES = None  # all labeled positives
SEED = 0  # exp19 sample
CONFIG_DIR = ROOT / "sections" / "model" / "configs"
CONFIG = Config.from_toml(
    CONFIG_DIR / "exp25_downsample_04" / "resnet18.toml", **OVERRIDES
)
EXP20 = Config.from_toml(CONFIG_DIR / "exp20_coarse_dropout_non_vip" / "resnet18.toml")
EXP16 = Config.from_toml(CONFIG_DIR / "exp16_downsampling" / "resnet18.toml")  # exp19 rows drawn from exp16 OOF
FIGURES = ROOT / "sections" / "model" / "figures"
PREFIX = "exp25_downsample_04"
SAVE_FIGURES = not SMOKE

# dataviz reference palette, light mode
SERIES = {"exp20": "#2a78d6", "downsample 0.4": "#1baf7a"}
plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "savefig.facecolor": "#fcfcfb",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#52514e",
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "axes.titlesize": "medium",
        "axes.titlecolor": "#52514e",
        "axes.grid": True,
        "grid.color": "#e1e0d9",
        "legend.frameon": False,
        "legend.fontsize": "small",
    }
)

mlflow.set_tracking_uri(MLFLOW_URI)
client = mlflow.MlflowClient()

## Training

In [ ]:
results = {"exp20": CVResult.load(EXP20)}
results["downsample 0.4"] = CrossValidator(CONFIG).run()
figures = {}

## Grad-CAM Cat, Dog: exp20 vs downsample 0.4

In [ ]:
cams = {"exp20": GradCamFigures(EXP20), "downsample 0.4": GradCamFigures(CONFIG)}
exp16 = GradCamFigures(EXP16)
for cls in ["Cat", "Dog"]:
    # exp19 rows; smoke OOF holds few rows
    rows = (cams["downsample 0.4"] if SMOKE else exp16).positive_rows(cls, N_SAMPLE, SEED)
    figures[f"{PREFIX}_grad_cam_{cls.lower()}_compare.png"] = compare(cams, cls, rows)
plt.show()

## Grad-CAM VIP: unseen vs trained images

In [ ]:
for cls in VIP:
    for train in (False, True):
        fig = cams["downsample 0.4"].positives(cls, train=train, n=N_POSITIVES)
        split = "train" if train else "validation"
        figures[f"{PREFIX}_grad_cam_{cls.lower()}_{split}.png"] = fig
plt.show()

## Precision-recall curves

In [ ]:
fig, axes = plt.subplots(
    1, len(CLASSES), figsize=(3.4 * len(CLASSES), 3.6), sharey=True
)
for cls, ax in zip(CLASSES, axes):
    for name, r in results.items():
        y, p = r.oof[cls], r.oof[f"p_{cls}"]
        precision, recall, _ = precision_recall_curve(y, p)
        ax.plot(
            recall,
            precision,
            color=SERIES[name],
            lw=2,
            drawstyle="steps-post",
            label=f"{name}, AP {average_precision_score(y, p):.3f}",
        )
    ax.set(
        title=f"{cls} ({int(y.sum())} positives)",
        xlabel="recall",
        xlim=(0, 1.02),
        ylim=(0, 1.05),
    )
    ax.legend(loc="lower left", alignment="left")
axes[0].set_ylabel("precision")
fig.tight_layout()
figures[f"{PREFIX}_pr_curves.png"] = fig
plt.show()

In [ ]:
with mlflow.start_run(
    experiment_id=client.get_experiment_by_name(CONFIG.experiment).experiment_id,
    run_name="summary",
):
    mlflow.log_param("val_mode", "fast")
    if SAVE_FIGURES:
        for fname, fig in figures.items():
            fig.savefig(FIGURES / fname, dpi=120)
            mlflow.log_figure(fig, fname)